# Pipeline de amostragem para dados desbalanceados — Bloco 1

Este bloco prepara as bases antes de qualquer técnica de amostragem.

| Célula | Conteúdo |
|---|---|
| 1 | Imports, parâmetros padrão e utilitários |
| 2 | Catálogo de métricas e validação das entradas |
| 3 | Diagnósticos: prevalência, decisão do módulo, chave, categóricas codificadas |
| 4 | Separação única por chave: treino, teste, OOT de seleção e OOT final |
| 5 | Seleção de features: amostras estratificadas do treino e votação sobre o `reduz_features` |
| 6 | Orquestração do bloco 1: `preparar_bases(...)` |
| 7 | Exemplo de uso |

Convenções:

- **Classe positiva**: alvo = 1 (a classe rara). **Classe negativa**: alvo = 0.
- Todas as funções recebem e devolvem DataFrames Spark, exceto as tabelas de relatório, que são DataFrames pandas.
- Nenhuma classe é usada; o estado do pipeline circula num dicionário chamado `contexto`.

## Célula 1 — Imports, parâmetros padrão e utilitários

`PARAMETROS_PADRAO` reúne todos os valores que o usuário pode sobrescrever. Os que pertencem aos blocos 2 e 3 (grade de técnicas, decisão, Optuna, relatório) já estão aqui para que a lista fique num só lugar.

`FORMATO_TABELA` fica como `"delta"`. As tabelas são gravadas no Unity Catalog como `catalogo.schema.<prefixo>_<nome>`.

In [ ]:
import sys
import importlib.metadata
from collections import Counter
from datetime import datetime

import numpy as np
import pandas as pd

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql import types as T

spark = SparkSession.getActiveSession()
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")

FORMATO_TABELA = "delta"

PARAMETROS_PADRAO = {
    # Ativação do módulo
    "desbalanceado": None,              # True: roda tudo | None: decide pela prevalência | False: não roda nada
    "limite_prevalencia": 0.05,         # com None, ativa o módulo se a prevalência do dev for menor que isso

    # Separação
    "proporcao_teste": 0.2,             # fração das chaves do dev que vai para o teste
    "proporcao_oot_selecao": 0.5,       # fração das chaves do OOT que vai para o OOT de seleção

    # Seleção de features (reduz_features)
    "n_amostras_reducao": 5,
    "negativos_por_positivo_reducao": 10,
    "max_positivos_reducao": 20000,     # acima disso, os positivos também são sorteados
    "votos_minimos": 3,
    "kwargs_reducao": {"tecnica": "lightgbm"},
    "limite_memoria_pandas_mb": 2000,   # estimativa máxima da amostra convertida para pandas

    # Diagnósticos
    "limite_categorica": 20,            # máximo de valores distintos para marcar uma coluna inteira como categórica codificada
    "minimo_positivos_aviso": 500,      # abaixo disso, o diagnóstico avisa que as métricas serão ruidosas
    "limite_positivos_driver": 100000,  # acima disso, técnicas que processam positivos no driver são excluídas

    # Grade de técnicas (bloco 2)
    "niveis_subamostragem": [5, 10, 20, 50],          # negativos por positivo
    "niveis_sobreamostragem": [2, 5, 10, 25],         # multiplicador dos positivos
    "niveis_peso": [25, 50, 100, 250],                # peso da classe positiva
    "niveis_negativos_confiaveis": [0.005, 0.01, 0.02],
    "sementes_grade": 3,                              # só para técnicas com sorteio
    "tamanho_lista_curta": 3,

    # Decisão e modelo final (bloco 3)
    "n_configuracoes_comuns": 5,
    "sementes_decisao": 2,
    "n_trials_optuna": 50,

    # Relatório
    "taxas_alerta_relatorio": [0.005, 0.01, 0.02, 0.05],
    "n_bootstrap": 500,

    # Gerais
    "semente": 42,
    "prefixo_tabelas": "pipeline_desbalanceado",
}


def log(mensagem):
    print(f"[{datetime.now():%H:%M:%S}] {mensagem}")


def coluna(nome):
    """Referência a uma coluna protegida por crases (nomes com ponto ou espaço)."""
    return F.col(f"`{nome}`")


def montar_parametros(**sobrescritos):
    """Junta os parâmetros padrão com os informados pelo usuário e recusa nomes desconhecidos."""
    desconhecidos = set(sobrescritos) - set(PARAMETROS_PADRAO)
    if desconhecidos:
        raise ValueError(f"Parâmetros desconhecidos: {sorted(desconhecidos)}")
    parametros = dict(PARAMETROS_PADRAO)
    parametros.update(sobrescritos)
    return parametros


def nome_tabela(destino_delta, prefixo, nome):
    return f"{destino_delta}.{prefixo}_{nome}"


def salvar_tabela(df, destino_delta, prefixo, nome, registro=None):
    """Grava o DataFrame como tabela e devolve a leitura da tabela gravada.

    Ler de volta da tabela materializa o resultado: sorteios feitos antes da gravação
    não são refeitos quando o DataFrame é usado de novo.
    """
    tabela = nome_tabela(destino_delta, prefixo, nome)
    (df.write.format(FORMATO_TABELA)
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(tabela))
    if registro is not None:
        registro["tabelas"].append(tabela)
    log(f"Tabela gravada: {tabela}")
    return spark.table(tabela)

## Célula 2 — Catálogo de métricas e validação das entradas

O catálogo liga cada string do AutoML à fórmula que ela calcula. Com o módulo ativo, só são aceitas as métricas que medem a ordenação dos scores:

| String | Fórmula | Módulo ativo |
|---|---|---|
| `ks` | KS (Kolmogorov-Smirnov) | Permitida |
| `roc`, `auc` | `areaUnderROC` (AUC-ROC) | Permitida |
| `pr_auc` | `areaUnderPR` (PR-AUC) | Permitida |
| `acuracia` | `accuracy` | Recusada |
| `precision recall` | `weightedPrecision` | Recusada |
| `recall` | `weightedRecall` (igual à acurácia em problema binário) | Recusada |

Se os nomes das strings no seu AutoML forem diferentes, ajuste só as chaves do dicionário.

In [ ]:
CATALOGO_METRICAS = {
    "ks":               {"formula": "ks",                "permitida_desbalanceado": True},
    "roc":              {"formula": "areaUnderROC",      "permitida_desbalanceado": True},
    "auc":              {"formula": "areaUnderROC",      "permitida_desbalanceado": True},
    "pr_auc":           {"formula": "areaUnderPR",       "permitida_desbalanceado": True},
    "acuracia":         {"formula": "accuracy",          "permitida_desbalanceado": False},
    "precision recall": {"formula": "weightedPrecision", "permitida_desbalanceado": False},
    "recall":           {"formula": "weightedRecall",    "permitida_desbalanceado": False},
}

TIPOS_NUMERICOS = (
    T.ByteType, T.ShortType, T.IntegerType, T.LongType,
    T.FloatType, T.DoubleType, T.DecimalType, T.BooleanType,
)


def resolver_features(df_dev, col_chave, col_safra, col_alvo, features=None):
    """Sem lista informada, usa todas as colunas exceto chave, safra e alvo."""
    if features is None:
        features = [c for c in df_dev.columns if c not in (col_chave, col_safra, col_alvo)]
    features = list(dict.fromkeys(features))  # remove repetidas mantendo a ordem
    if not features:
        raise ValueError("Nenhuma feature encontrada.")
    return features


def validar_entradas(df_dev, df_oot, col_chave, col_safra, col_alvo, features, metrica):
    """Confere colunas, tipos, métrica conhecida e alvo binário. Levanta erro com a lista de problemas."""
    erros = []
    obrigatorias = [col_chave, col_safra, col_alvo] + features
    for nome, df in (("df_dev", df_dev), ("df_oot", df_oot)):
        faltando = [c for c in obrigatorias if c not in df.columns]
        if faltando:
            erros.append(f"{nome}: {len(faltando)} colunas ausentes, por exemplo {faltando[:10]}")

    campos = {f.name: f.dataType for f in df_dev.schema.fields}
    nao_numericas = [c for c in features if c in campos and not isinstance(campos[c], TIPOS_NUMERICOS)]
    if nao_numericas:
        erros.append(f"Features não numéricas (strings não são aceitas): {nao_numericas[:10]}")

    if metrica not in CATALOGO_METRICAS:
        erros.append(f"Métrica '{metrica}' desconhecida. Opções: {sorted(CATALOGO_METRICAS)}")

    if erros:
        raise ValueError("Entradas inválidas:\n- " + "\n- ".join(erros))

    for nome, df in (("df_dev", df_dev), ("df_oot", df_oot)):
        valores = {r[0] for r in df.select(coluna(col_alvo)).distinct().limit(5).collect()}
        if not valores <= {0, 1}:
            raise ValueError(f"{nome}: o alvo deve conter só 0 e 1, sem nulos. Valores encontrados: {sorted(valores, key=str)}")


def validar_metrica(metrica, modulo_ativo):
    """Com o módulo ativo, recusa as métricas que não medem a ordenação dos scores."""
    info = CATALOGO_METRICAS[metrica]
    if modulo_ativo and not info["permitida_desbalanceado"]:
        permitidas = sorted(m for m, i in CATALOGO_METRICAS.items() if i["permitida_desbalanceado"])
        raise ValueError(
            f"A métrica '{metrica}' ({info['formula']}) não serve para alvo raro: "
            f"um modelo que nunca alerta já obtém valor próximo do máximo. Use uma de {permitidas}."
        )


def versao_pacote(nome_distribuicao):
    try:
        return importlib.metadata.version(nome_distribuicao)
    except importlib.metadata.PackageNotFoundError:
        return "não encontrada"


def iniciar_registro(parametros, col_chave, col_safra, col_alvo, metrica, features):
    return {
        "inicio": datetime.now().isoformat(timespec="seconds"),
        "parametros": dict(parametros),
        "colunas": {"chave": col_chave, "safra": col_safra, "alvo": col_alvo},
        "metrica": metrica,
        "formula_metrica": CATALOGO_METRICAS[metrica]["formula"],
        "n_features_iniciais": len(features),
        "versoes": {
            "python": sys.version.split()[0],
            "spark": spark.version,
            "pandas": pd.__version__,
            "numpy": np.__version__,
            "synapseml": versao_pacote("synapseml"),
            "lightgbm_python": versao_pacote("lightgbm"),
        },
        "tabelas": [],
        "avisos": [],
    }

## Célula 3 — Diagnósticos

- `calcular_prevalencia`: linhas, positivos, negativos e prevalência.
- `decidir_modulo`: aplica a regra do parâmetro `desbalanceado`. Com `None`, o módulo é ativado se a prevalência do dev for menor que `limite_prevalencia`.
- `diagnosticar_chave`: verifica se a chave se repete, se aparece em várias safras, se o alvo muda dentro da mesma chave e quantas chaves existem tanto no dev quanto no OOT.
- `contar_por_safra`: linhas, positivos e prevalência por safra.
- `detectar_categoricas`: marca como categórica codificada a coluna que só tem valores inteiros e no máximo `limite_categorica` valores distintos. Nulos e NaN (*not a number*, valor numérico ausente) são ignorados. A contagem de distintos usa `approx_count_distinct` (contagem aproximada, exata para cardinalidades pequenas como essas), num único passo sobre a base. O contexto guarda só as categóricas que sobreviveram à seleção de features; a lista completa fica em `registro["categoricas_detectadas"]`.

In [ ]:
def calcular_prevalencia(df, col_alvo):
    r = df.agg(
        F.count(F.lit(1)).alias("linhas"),
        F.sum(coluna(col_alvo).cast("long")).alias("positivos"),
    ).first()
    linhas = int(r["linhas"])
    positivos = int(r["positivos"] or 0)
    return {
        "linhas": linhas,
        "positivos": positivos,
        "negativos": linhas - positivos,
        "prevalencia": positivos / linhas if linhas else float("nan"),
    }


def decidir_modulo(desbalanceado, prevalencia, limite_prevalencia):
    if desbalanceado is True:
        return True, "Módulo ativado pelo usuário (desbalanceado=True)."
    if desbalanceado is False:
        return False, "Módulo desligado pelo usuário (desbalanceado=False)."
    if prevalencia < limite_prevalencia:
        return True, (f"Módulo ativado pelo diagnóstico: prevalência do dev de {prevalencia:.4%}, "
                      f"abaixo do limite de {limite_prevalencia:.2%}.")
    return False, (f"Módulo não ativado: prevalência do dev de {prevalencia:.4%}, "
                   f"igual ou acima do limite de {limite_prevalencia:.2%}.")


def diagnosticar_chave(df_dev, df_oot, col_chave, col_safra, col_alvo):
    resumo = {}
    for nome, df in (("dev", df_dev), ("oot", df_oot)):
        por_chave = df.groupBy(coluna(col_chave)).agg(
            F.count(F.lit(1)).alias("linhas"),
            F.countDistinct(coluna(col_safra)).alias("safras"),
            F.countDistinct(coluna(col_alvo)).alias("alvos"),
        )
        r = por_chave.agg(
            F.count(F.lit(1)).alias("chaves"),
            F.sum("linhas").alias("linhas"),
            F.max("linhas").alias("max_linhas_por_chave"),
            F.sum(F.when(F.col("safras") > 1, 1).otherwise(0)).alias("chaves_em_varias_safras"),
            F.sum(F.when(F.col("alvos") > 1, 1).otherwise(0)).alias("chaves_com_alvo_variavel"),
        ).first().asDict()
        r = {k: int(v or 0) for k, v in r.items()}
        r["chave_repete"] = r["linhas"] > r["chaves"]
        resumo[nome] = r
    resumo["chaves_comuns_dev_oot"] = (
        df_dev.select(coluna(col_chave)).distinct()
        .join(df_oot.select(coluna(col_chave)).distinct(), on=col_chave, how="inner")
        .count()
    )
    return resumo


def contar_por_safra(df, col_safra, col_alvo):
    return (
        df.groupBy(coluna(col_safra))
          .agg(F.count(F.lit(1)).alias("linhas"),
               F.sum(coluna(col_alvo).cast("long")).alias("positivos"))
          .withColumn("prevalencia", F.col("positivos") / F.col("linhas"))
          .orderBy(col_safra)
          .toPandas()
    )


def detectar_categoricas(df, features, limite_categorica):
    """Coluna só com inteiros e no máximo `limite_categorica` valores distintos -> categórica codificada."""
    expressoes = []
    for i, c in enumerate(features):
        bruto = coluna(c).cast("double")
        valor = F.when(~F.isnan(bruto), bruto)  # NaN tratado como ausente, igual a nulo
        expressoes.append(F.approx_count_distinct(valor, rsd=0.01).alias(f"d{i}"))
        expressoes.append(
            F.max(F.when(valor.isNotNull() & (valor != F.floor(valor)), 1).otherwise(0)).alias(f"f{i}")
        )
    r = df.agg(*expressoes).first()
    return [c for i, c in enumerate(features) if r[f"f{i}"] == 0 and r[f"d{i}"] <= limite_categorica]

## Célula 4 — Separação única por chave

A separação é feita no nível da chave: todas as linhas de uma mesma chave vão para a mesma parte.

1. Cada chave recebe um estrato: a primeira safra em que aparece e se ela foi positiva em alguma linha.
2. Cada chave recebe um número entre 0 e 1 calculado por *hash* (função que transforma a chave num número fixo) da chave com a semente. O resultado é o mesmo em qualquer execução, ao contrário de `rand()`, que depende do particionamento.
3. Dentro de cada estrato, as chaves são ordenadas por esse número, e as primeiras vão para a segunda parte (teste, ou OOT final) até completar a proporção. Isso garante a mesma proporção de positivos e de safras em cada parte.

O dev vira **treino** e **teste**; o OOT vira **OOT de seleção** e **OOT final**, com sementes diferentes.

O resultado é gravado como a tabela `mapa_separacao` (base, chave, parte). As quatro partes são reconstruídas com *left semi join* (junção que só filtra as linhas cujas chaves estão no mapa, sem trazer colunas dele), sem duplicar os dados.

In [ ]:
def uniforme_por_chave(col_chave, semente):
    """Número entre 0 e 1 fixo para cada chave e semente."""
    h = F.xxhash64(coluna(col_chave).cast("string"), F.lit(int(semente)))
    return F.pmod(h, F.lit(1_000_000_007)).cast("double") / F.lit(1_000_000_007.0)


def mapear_por_chave(df, col_chave, col_safra, col_alvo, fracao_segunda, nome_primeira, nome_segunda, semente):
    """Atribui cada chave a uma de duas partes, estratificando por primeira safra x positivo."""
    por_chave = (
        df.groupBy(coluna(col_chave))
          .agg(F.min(coluna(col_safra)).alias("_estrato_safra"),
               F.max(coluna(col_alvo).cast("int")).alias("_estrato_alvo"))
          .withColumn("_u", uniforme_por_chave(col_chave, semente))
    )
    ordem = Window.partitionBy("_estrato_safra", "_estrato_alvo").orderBy("_u", col_chave)
    estrato = Window.partitionBy("_estrato_safra", "_estrato_alvo")
    return (
        por_chave
        .withColumn("_posicao", F.row_number().over(ordem))
        .withColumn("_total", F.count(F.lit(1)).over(estrato))
        .withColumn(
            "parte",
            F.when(F.col("_posicao") <= F.round(F.col("_total") * F.lit(float(fracao_segunda))), F.lit(nome_segunda))
             .otherwise(F.lit(nome_primeira)),
        )
        .select(coluna(col_chave), "parte")
    )


def separar_bases(df_dev, df_oot, col_chave, col_safra, col_alvo, parametros, destino_delta, registro):
    semente = parametros["semente"]
    mapa_dev = mapear_por_chave(
        df_dev, col_chave, col_safra, col_alvo,
        fracao_segunda=parametros["proporcao_teste"],
        nome_primeira="treino", nome_segunda="teste", semente=semente,
    ).withColumn("base", F.lit("dev"))
    mapa_oot = mapear_por_chave(
        df_oot, col_chave, col_safra, col_alvo,
        fracao_segunda=1 - parametros["proporcao_oot_selecao"],
        nome_primeira="oot_selecao", nome_segunda="oot_final", semente=semente + 1,
    ).withColumn("base", F.lit("oot"))

    mapa = salvar_tabela(
        mapa_dev.unionByName(mapa_oot).select("base", coluna(col_chave), "parte"),
        destino_delta, parametros["prefixo_tabelas"], "mapa_separacao", registro,
    )

    origem = {"treino": df_dev, "teste": df_dev, "oot_selecao": df_oot, "oot_final": df_oot}
    partes = {}
    for parte, df in origem.items():
        chaves = mapa.filter(F.col("parte") == parte).select(coluna(col_chave))
        partes[parte] = df.join(chaves, on=col_chave, how="left_semi")
    return partes, mapa


def resumir_separacao(partes, col_safra, col_alvo):
    """Linhas, positivos e prevalência de cada parte, por safra e no total."""
    tabelas = []
    for nome, df in partes.items():
        por_safra = contar_por_safra(df, col_safra, col_alvo)
        por_safra.insert(0, "parte", nome)
        total = pd.DataFrame({
            "parte": [nome], col_safra: ["TOTAL"],
            "linhas": [por_safra["linhas"].sum()], "positivos": [por_safra["positivos"].sum()],
        })
        total["prevalencia"] = total["positivos"] / total["linhas"]
        por_safra[col_safra] = por_safra[col_safra].astype(str)
        tabelas.extend([por_safra, total])
    return pd.concat(tabelas, ignore_index=True)

## Célula 5 — Seleção de features com o `reduz_features`

Para cada uma das `n_amostras_reducao` rodadas:

1. Do **treino** (nunca do teste ou do OOT), são mantidos todos os positivos (ou um sorteio deles, se passarem de `max_positivos_reducao`) e sorteados `negativos_por_positivo_reducao` negativos por positivo, estratificados por safra. Cada rodada usa uma semente diferente, e por isso recebe negativos diferentes.
2. Só as features e o alvo são selecionados, e a amostra é convertida para pandas.
3. A função do usuário é chamada como `funcao_reducao(amostra_pandas, **kwargs_reducao)` e deve devolver uma lista de nomes de features.

Depois das rodadas, cada feature recebe um voto por lista em que apareceu, e ficam as que tiverem pelo menos `votos_minimos` votos (seleção por estabilidade).

Antes da primeira conversão, a função estima a memória da amostra em pandas (linhas × colunas × 8 bytes) e interrompe se passar de `limite_memoria_pandas_mb`.

Se o `reduz_features` espera o alvo com outro nome, ou outros argumentos, informe-os em `kwargs_reducao`.

In [ ]:
def sortear_amostra_reducao(df_treino, features, col_alvo, col_safra, contagem, safras,
                            negativos_por_positivo, max_positivos, semente):
    positivos = df_treino.filter(coluna(col_alvo) == 1)
    n_pos = contagem["positivos"]
    if n_pos > max_positivos:
        positivos = positivos.sample(withReplacement=False, fraction=max_positivos / n_pos, seed=semente)
        n_pos = max_positivos
    fracao_neg = min(1.0, n_pos * negativos_por_positivo / max(contagem["negativos"], 1))
    negativos = (df_treino.filter(coluna(col_alvo) == 0)
                 .sampleBy(col_safra, fractions={s: fracao_neg for s in safras}, seed=semente))
    return positivos.unionByName(negativos).select(*[coluna(c) for c in features], coluna(col_alvo))


def selecionar_features(df_treino, features, col_alvo, col_safra, funcao_reducao, parametros, registro):
    n_amostras = parametros["n_amostras_reducao"]
    votos_minimos = parametros["votos_minimos"]
    if not 1 <= votos_minimos <= n_amostras:
        raise ValueError(f"votos_minimos ({votos_minimos}) deve estar entre 1 e n_amostras_reducao ({n_amostras}).")

    contagem = calcular_prevalencia(df_treino, col_alvo)
    if contagem["positivos"] == 0:
        raise ValueError("O treino não tem positivos.")
    safras = [r[0] for r in df_treino.select(coluna(col_safra)).distinct().collect()]

    n_pos_previstos = min(contagem["positivos"], parametros["max_positivos_reducao"])
    linhas_previstas = n_pos_previstos * (1 + parametros["negativos_por_positivo_reducao"])
    memoria_mb = linhas_previstas * (len(features) + 1) * 8 / 1e6
    log(f"Amostra por rodada: cerca de {linhas_previstas:,} linhas, {memoria_mb:,.0f} MB em pandas.")
    if memoria_mb > parametros["limite_memoria_pandas_mb"]:
        raise MemoryError(
            f"A amostra estimada ({memoria_mb:,.0f} MB) passa de limite_memoria_pandas_mb "
            f"({parametros['limite_memoria_pandas_mb']} MB). Reduza negativos_por_positivo_reducao "
            f"ou max_positivos_reducao."
        )

    votos = Counter()
    rodadas = []
    for i in range(n_amostras):
        semente_i = parametros["semente"] + 1000 * (i + 1)
        amostra = sortear_amostra_reducao(
            df_treino, features, col_alvo, col_safra, contagem, safras,
            parametros["negativos_por_positivo_reducao"], parametros["max_positivos_reducao"], semente_i,
        ).toPandas()
        lista = list(funcao_reducao(amostra, **parametros["kwargs_reducao"]))
        desconhecidas = sorted(set(lista) - set(features))
        if desconhecidas:
            raise ValueError(f"A função de redução devolveu nomes fora da lista de features: {desconhecidas[:10]}")
        votos.update(set(lista))
        rodadas.append({
            "rodada": i + 1, "semente": semente_i, "linhas": len(amostra),
            "positivos": int(amostra[col_alvo].sum()), "features_escolhidas": len(lista),
        })
        log(f"Rodada {i + 1}/{n_amostras}: {len(amostra):,} linhas, "
            f"{rodadas[-1]['positivos']:,} positivos, {len(lista)} features escolhidas.")
        del amostra

    estabilidade = pd.DataFrame({"feature": features, "votos": [votos.get(f, 0) for f in features]})
    estabilidade["mantida"] = estabilidade["votos"] >= votos_minimos
    estabilidade = estabilidade.sort_values(["votos", "feature"], ascending=[False, True]).reset_index(drop=True)
    selecionadas = estabilidade.loc[estabilidade["mantida"], "feature"].tolist()
    if not selecionadas:
        raise ValueError("Nenhuma feature atingiu votos_minimos. Reveja a função de redução ou reduza votos_minimos.")
    log(f"Features mantidas: {len(selecionadas)} de {len(features)} (mínimo de {votos_minimos} votos em {n_amostras}).")
    return selecionadas, estabilidade, pd.DataFrame(rodadas)

## Célula 6 — Orquestração do bloco 1

`preparar_bases(...)` executa as células 2 a 5 na ordem e devolve o dicionário `contexto`, que os blocos 2 e 3 vão receber.

| Situação | O que executa |
|---|---|
| `desbalanceado=False` | Nada. Devolve `{"modulo_ativo": False, "motivo": ...}` sem ler os dados |
| `desbalanceado=None` e prevalência do dev ≥ limite | Só a validação e o cálculo da prevalência |
| `desbalanceado=True`, ou `None` e prevalência < limite | O bloco 1 completo |

As partes `treino` e `teste` são mantidas em cache (`persist`), porque serão lidas muitas vezes no bloco 2. Use `liberar_cache(contexto)` ao terminar.

Chaves do `contexto`: `modulo_ativo`, `motivo`, `parametros`, `registro`, `colunas`, `metrica`, `prevalencia_dev`, `diagnostico_chave`, `safras_dev`, `safras_oot`, `separacao`, `categoricas`, `features_iniciais`, `features_selecionadas`, `estabilidade_features`, `rodadas_reducao`, `partes`, `destino_delta`.

In [ ]:
def preparar_bases(df_dev, df_oot, col_chave, col_safra, col_alvo, metrica, funcao_reducao,
                   destino_delta, features=None, **sobrescritos):
    parametros = montar_parametros(**sobrescritos)

    if parametros["desbalanceado"] is False:
        return {"modulo_ativo": False, "motivo": "Módulo desligado pelo usuário (desbalanceado=False)."}

    # Validação e decisão do módulo
    features = resolver_features(df_dev, col_chave, col_safra, col_alvo, features)
    validar_entradas(df_dev, df_oot, col_chave, col_safra, col_alvo, features, metrica)
    prevalencia_dev = calcular_prevalencia(df_dev, col_alvo)
    ativo, motivo = decidir_modulo(parametros["desbalanceado"], prevalencia_dev["prevalencia"],
                                   parametros["limite_prevalencia"])
    log(motivo)
    if not ativo:
        return {"modulo_ativo": False, "motivo": motivo, "prevalencia_dev": prevalencia_dev}
    validar_metrica(metrica, ativo)

    registro = iniciar_registro(parametros, col_chave, col_safra, col_alvo, metrica, features)
    registro["motivo_modulo"] = motivo

    # Diagnósticos
    log("Diagnosticando a chave.")
    diagnostico_chave = diagnosticar_chave(df_dev, df_oot, col_chave, col_safra, col_alvo)
    if diagnostico_chave["dev"]["chave_repete"]:
        log("A chave se repete no dev: separação e amostragens serão feitas por chave.")
    if diagnostico_chave["dev"]["chaves_com_alvo_variavel"] > 0:
        registro["avisos"].append(
            f"{diagnostico_chave['dev']['chaves_com_alvo_variavel']:,} chaves do dev têm alvo 0 e 1 em linhas diferentes; "
            "na separação, elas contam como positivas."
        )
    safras_dev = contar_por_safra(df_dev, col_safra, col_alvo)
    safras_oot = contar_por_safra(df_oot, col_safra, col_alvo)

    # Separação única
    log("Separando treino, teste, OOT de seleção e OOT final.")
    partes, _ = separar_bases(df_dev, df_oot, col_chave, col_safra, col_alvo, parametros, destino_delta, registro)
    separacao = resumir_separacao(partes, col_safra, col_alvo)

    positivos_treino = int(separacao.query("parte == 'treino' and `{}` == 'TOTAL'".format(col_safra))["positivos"].iloc[0])
    if positivos_treino < parametros["minimo_positivos_aviso"]:
        registro["avisos"].append(
            f"O treino tem só {positivos_treino:,} positivos: as métricas e a comparação entre técnicas serão ruidosas."
        )
    for parte in ("teste", "oot_selecao", "oot_final"):
        n = int(separacao.query("parte == @parte and `{}` == 'TOTAL'".format(col_safra))["positivos"].iloc[0])
        if n < parametros["minimo_positivos_aviso"]:
            registro["avisos"].append(f"A parte {parte} tem só {n:,} positivos.")

    # Categóricas codificadas (no treino)
    log("Detectando categóricas codificadas no treino.")
    categoricas = detectar_categoricas(partes["treino"], features, parametros["limite_categorica"])
    registro["categoricas_detectadas"] = list(categoricas)
    log(f"{len(categoricas)} colunas detectadas como categóricas codificadas.")

    # Seleção de features
    log("Selecionando features com o reduz_features.")
    selecionadas, estabilidade, rodadas = selecionar_features(
        partes["treino"], features, col_alvo, col_safra, funcao_reducao, parametros, registro,
    )
    categoricas = [c for c in categoricas if c in selecionadas]

    colunas_finais = [col_chave, col_safra, col_alvo] + selecionadas
    partes = {nome: df.select(*[coluna(c) for c in colunas_finais]) for nome, df in partes.items()}
    partes["treino"] = partes["treino"].persist()
    partes["teste"] = partes["teste"].persist()

    for aviso in registro["avisos"]:
        log(f"AVISO: {aviso}")

    return {
        "modulo_ativo": True,
        "motivo": motivo,
        "parametros": parametros,
        "registro": registro,
        "colunas": {"chave": col_chave, "safra": col_safra, "alvo": col_alvo},
        "metrica": metrica,
        "prevalencia_dev": prevalencia_dev,
        "diagnostico_chave": diagnostico_chave,
        "safras_dev": safras_dev,
        "safras_oot": safras_oot,
        "separacao": separacao,
        "categoricas": categoricas,
        "features_iniciais": features,
        "features_selecionadas": selecionadas,
        "estabilidade_features": estabilidade,
        "rodadas_reducao": rodadas,
        "partes": partes,
        "destino_delta": destino_delta,
    }


def liberar_cache(contexto):
    for df in contexto.get("partes", {}).values():
        df.unpersist()

## Célula 7 — Exemplo de uso

Substitua os nomes das tabelas, das colunas e do destino. O `reduz_features` deve estar definido no notebook (por exemplo, com `%run` do notebook onde ele está).

In [ ]:
# df_dev = spark.table("catalogo.schema.base_desenvolvimento")
# df_oot = spark.table("catalogo.schema.base_oot")
#
# contexto = preparar_bases(
#     df_dev=df_dev,
#     df_oot=df_oot,
#     col_chave="id_conta",
#     col_safra="safra",
#     col_alvo="alvo",
#     metrica="ks",
#     funcao_reducao=reduz_features,
#     destino_delta="catalogo.schema",
#     desbalanceado=None,
#     kwargs_reducao={"tecnica": "lightgbm"},
# )
#
# if contexto["modulo_ativo"]:
#     display(contexto["separacao"])
#     display(contexto["estabilidade_features"])
#     print(contexto["diagnostico_chave"])
#     print(contexto["categoricas"])
#     print(contexto["registro"]["avisos"])
# else:
#     print(contexto["motivo"])